<a href="https://colab.research.google.com/github/EkaterinaLavlinskaya/GPT-Learning-Language-Mini-Model/blob/main/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GPT упрощенная

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import Dataset, DataLoader

In [ ]:
# Маленький текст для обучения
text = "hello world hello pytorch hello transformer hello world hello pytorch hello transformer hello world hello pytorch"
chars = sorted(list(set(text)))
stoi = {ch: i for i, ch in enumerate(chars)}  # символ -> индекс
itos = {i: ch for i, ch in enumerate(chars)}  # индекс -> символ
vocab_size = len(chars)

In [ ]:
# Создаём последовательности для обучения
block_size = 8  # длина контекста (сколько символов смотрим)
X = []
y = []
for i in range(len(text) - block_size):
    X.append([stoi[ch] for ch in text[i:i+block_size]])
    y.append(stoi[text[i+block_size]])

X = torch.tensor(X)
y = torch.tensor(y)

print(f"Символов: {vocab_size}")
print(f"Пример: '{text[:block_size]}' -> следующий '{text[block_size]}'")

Символов: 17
Пример: 'hello wo' -> следующий 'r'


In [ ]:
# Модель упрощенная GPT

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=6):
        super().__init__()
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)

        # Слой трансформера (вместо целого декодера для простоты)
        decoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=256,
            dropout=0.1,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x: [batch, seq_len]
        batch_size, seq_len = x.shape

        # Эмбеддинги + позиции
        token_emb = self.token_embedding(x)  # [batch, seq_len, d_model]
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        pos_emb = self.position_embedding(positions)  # [1, seq_len, d_model]
        x = token_emb + pos_emb

        # Маска, чтобы не видеть будущее
        mask = torch.triu(torch.ones(seq_len, seq_len) * float('-inf'), diagonal=1)
        mask = mask.to(x.device)

        # Трансформер
        x = self.transformer(x, mask=mask)

        # Предсказание следующего символа
        logits = self.lm_head(x)  # [batch, seq_len, vocab_size]
        return logits

    def generate(self, start_text, max_new_tokens=20):
        """Генерация текста"""
        self.eval()
        with torch.no_grad():
            # Переводим начало в индексы
            tokens = [stoi[ch] for ch in start_text]

            for _ in range(max_new_tokens):
                # Берём последние block_size символов
                if len(tokens) > block_size:
                    input_tokens = tokens[-block_size:]
                else:
                    input_tokens = tokens

                # Превращаем в тензор
                x = torch.tensor(input_tokens).unsqueeze(0)

                # Получаем предсказания
                logits = self(x)

                # Берём последний токен
                next_token_logits = logits[0, -1, :]

                # Выбираем следующий токен
                probs = F.softmax(next_token_logits / 0.8, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1).item()

                tokens.append(next_token)

            # Превращаем обратно в текст
            generated = ''.join([itos[t] for t in tokens])
            return generated


In [ ]:
# Обучение

model = MiniGPT(vocab_size, d_model=128, nhead=4, num_layers=6)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# DataLoader
dataset = torch.utils.data.TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print("\nНачинаем обучение...")
for epoch in range(50):
    total_loss = 0
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()

        # Прямой проход
        logits = model(x_batch)  # [batch, seq_len, vocab_size]

        # Берём предсказания для последней позиции
        logits_flat = logits[:, :-1, :].reshape(-1, vocab_size) # [batch, vocab_size]
        targets_flat = x_batch[:, 1:].reshape(-1)
        # Считаем loss
        loss = criterion(logits_flat, targets_flat)

        # Обратный проход
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Эпоха {epoch}, Loss: {total_loss/len(dataloader):.5f}")


Начинаем обучение...
Эпоха 0, Loss: 2.36256
Эпоха 10, Loss: 0.24626
Эпоха 20, Loss: 0.23481
Эпоха 30, Loss: 0.21988
Эпоха 40, Loss: 0.20930


In [ ]:
# Тестируем

print("\n" + "="*50)
print("Генерация текста:")
print("-" * 50)

test_starts = ["h", "he", "hel", "hell", "hello"]
for start in test_starts:
    generated = model.generate(start, max_new_tokens=15)
    print(f"'{start}' -> '{generated}'")


Генерация текста:
--------------------------------------------------
'h' -> 'hello pytratorch'
'he' -> 'hello pytrcheld h'
'hel' -> 'hello transformer '
'hell' -> 'hello pytransformer'
'hello' -> 'hello pytransformer '
